In [9]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from Utils.Accuracy_measures import topk_accuracy
from Utils.TinyImageNet_loader import get_tinyimagenet_dataloaders
from Utils.Num_parameter import count_parameters
from Models.Resnet50 import Resnet50

import torchvision.transforms as transforms
from torch import nn
from torch import optim

import time
import torch
import os

In [19]:
device = 'cpu'

In [23]:
model = Resnet50(pretrained=False,
                          weights_path='../weights/resnet50_weights.pth',
                          tensorized=False,
                          input_shape=(192,192),
                          num_classes=200,
                          avg_pool=False,
                          new_classifier=None).to(device)

In [17]:
image_size = 192

tiny_transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(64, padding=4),
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
tiny_transform_val = transforms.Compose([
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
tiny_transform_test = transforms.Compose([
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])


tiny_train_loader, tiny_val_loader, tiny_test_loader = get_tinyimagenet_dataloaders(data_dir = '../datasets',
                                                                                    transform_train=tiny_transform_train,
                                                                                    transform_val=tiny_transform_val,
                                                                                    transform_test=tiny_transform_test,
                                                                                    batch_size=10,
                                                                                    image_size=192)

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [25]:
def test(loader, epoch):
    model.eval()
    
    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
            
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)
    
    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [26]:
test(tiny_train_loader, 1)

In [ ]:
test(tiny_test_loader, 1)